In [1]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

mysql_username = input(
    "MySQL username [root]: "
).strip() or "root"

mysql_password = getpass(
    "Enter your MySQL password: "
)

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_username,
    password=mysql_password,
    host="localhost",
    port=3306,
    database="cart2insights"
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

with engine.connect() as connection:
    print(
        connection.execute(
            text("SELECT DATABASE()")
        ).scalar()
    )

cart2insights


In [3]:
def run_query(query):
    return pd.read_sql(
        text(query),
        con=engine
    )

In [4]:
overview_query = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
    ROUND(SUM(p.payment_value), 2) AS total_revenue,
    ROUND(
        SUM(p.payment_value) /
        COUNT(DISTINCT o.order_id),
        2
    ) AS average_order_value
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
INNER JOIN payments AS p
    ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';
"""

overview_df = run_query(overview_query)
overview_df

,total_orders,unique_customers,total_revenue,average_order_value
0,96477,93357,15422461.77,159.86


In [5]:
monthly_sales_query = """
SELECT
    DATE_FORMAT(
        o.order_purchase_timestamp,
        '%Y-%m'
    ) AS order_month,

    COUNT(DISTINCT o.order_id) AS total_orders,

    ROUND(
        SUM(p.payment_value),
        2
    ) AS total_revenue,

    ROUND(
        SUM(p.payment_value) /
        COUNT(DISTINCT o.order_id),
        2
    ) AS average_order_value

FROM orders AS o

INNER JOIN payments AS p
    ON o.order_id = p.order_id

WHERE o.order_status = 'delivered'

GROUP BY
    DATE_FORMAT(
        o.order_purchase_timestamp,
        '%Y-%m'
    )

ORDER BY order_month;
"""

monthly_sales_df = run_query(monthly_sales_query)
monthly_sales_df

,order_month,total_orders,total_revenue,average_order_value
0,2016-10,265,46566.71,175.72
1,2016-12,1,19.62,19.62
2,2017-01,750,127545.67,170.06
3,2017-02,1653,271298.65,164.13
4,2017-03,2546,414369.39,162.75
5,2017-04,2303,390952.18,169.76
6,2017-05,3546,567066.73,159.92
7,2017-06,3135,490225.60,156.37
8,2017-07,3872,566403.93,146.28
9,2017-08,4193,646000.61,154.07


In [6]:

category_performance_query = """
SELECT
    COALESCE(
        ct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category_name,

    COUNT(oi.order_item_id) AS items_sold,

    COUNT(DISTINCT oi.order_id) AS total_orders,

    ROUND(
        SUM(oi.price),
        2
    ) AS product_revenue,

    ROUND(
        AVG(oi.price),
        2
    ) AS average_item_price

FROM order_items AS oi

INNER JOIN orders AS o
    ON oi.order_id = o.order_id

INNER JOIN products AS p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation AS ct
    ON p.product_category_name =
       ct.product_category_name

WHERE o.order_status = 'delivered'

GROUP BY category_name

ORDER BY product_revenue DESC

LIMIT 10;
"""

category_performance_df = run_query(
    category_performance_query
)

category_performance_df

,category_name,items_sold,total_orders,product_revenue,average_item_price
0,health_beauty,9465,8647,1233131.72,130.28
1,watches_gifts,5859,5495,1166176.98,199.04
2,bed_bath_table,10953,9272,1023434.76,93.44
3,sports_leisure,8431,7530,954852.55,113.25
4,computers_accessories,7644,6530,888724.61,116.26
5,furniture_decor,8160,6307,711927.69,87.25
6,housewares,6795,5743,615628.69,90.60
7,cool_stuff,3718,3559,610204.10,164.12
8,auto,4140,3810,578966.65,139.85
9,toys,4030,3804,471286.48,116.94


In [7]:
state_performance_query = """
SELECT
    c.customer_state AS state,

    COUNT(
        DISTINCT c.customer_unique_id
    ) AS unique_customers,

    COUNT(
        DISTINCT o.order_id
    ) AS total_orders,

    ROUND(
        SUM(p.payment_value),
        2
    ) AS total_revenue,

    ROUND(
        SUM(p.payment_value) /
        COUNT(DISTINCT o.order_id),
        2
    ) AS average_order_value

FROM customers AS c

INNER JOIN orders AS o
    ON c.customer_id = o.customer_id

INNER JOIN payments AS p
    ON o.order_id = p.order_id

WHERE o.order_status = 'delivered'

GROUP BY c.customer_state

ORDER BY total_revenue DESC

LIMIT 10;
"""

state_performance_df = run_query(
    state_performance_query
)

state_performance_df

,state,unique_customers,total_orders,total_revenue,average_order_value
0,SP,39155,40500,5770266.19,142.48
1,RJ,11917,12350,2055690.45,166.45
2,MG,11001,11354,1819277.61,160.23
3,RS,5168,5345,861802.40,161.24
4,PR,4769,4923,781919.55,158.83
5,SC,3449,3546,595208.40,167.85
6,BA,3158,3256,591270.60,181.59
7,DF,2019,2080,346146.17,166.42
8,GO,1895,1957,334294.22,170.82
9,ES,1928,1995,317682.65,159.24


In [8]:
delivery_performance_query = """
SELECT
    c.customer_state AS state,

    COUNT(o.order_id) AS delivered_orders,

    ROUND(
        AVG(
            DATEDIFF(
                o.order_delivered_customer_date,
                o.order_purchase_timestamp
            )
        ),
        2
    ) AS average_delivery_days,

    SUM(
        CASE
            WHEN o.order_delivered_customer_date >
                 o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date >
                     o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(o.order_id),
        2
    ) AS late_delivery_percentage

FROM orders AS o

INNER JOIN customers AS c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL

GROUP BY c.customer_state

ORDER BY late_delivery_percentage DESC;
"""

delivery_performance_df = run_query(
    delivery_performance_query
)

delivery_performance_df

,state,delivered_orders,average_delivery_days,late_orders,late_delivery_percentage
0,AL,397,24.50,95.0,23.93
1,MA,717,21.51,141.0,19.67
2,PI,476,19.39,76.0,15.97
3,CE,1279,21.20,196.0,15.32
4,SE,335,21.46,51.0,15.22
5,BA,3256,19.28,457.0,14.04
6,RJ,12350,15.24,1664.0,13.47
7,TO,274,17.60,35.0,12.77
8,PA,946,23.73,117.0,12.37
9,ES,1995,15.72,244.0,12.23


In [9]:
delivery_review_query = """
SELECT
    CASE
        WHEN o.order_delivered_customer_date >
             o.order_estimated_delivery_date
        THEN 'Late'
        ELSE 'On Time or Early'
    END AS delivery_status,

    COUNT(r.review_id) AS total_reviews,

    ROUND(
        AVG(r.review_score),
        2
    ) AS average_review_score,

    SUM(
        CASE
            WHEN r.review_score <= 2
            THEN 1
            ELSE 0
        END
    ) AS low_score_reviews,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN r.review_score <= 2
                THEN 1
                ELSE 0
            END
        ) / COUNT(r.review_id),
        2
    ) AS low_score_percentage

FROM orders AS o

INNER JOIN reviews AS r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL

GROUP BY delivery_status

ORDER BY delivery_status;
"""

delivery_review_df = run_query(
    delivery_review_query
)

delivery_review_df

,delivery_status,total_reviews,average_review_score,low_score_reviews,low_score_percentage
0,Late,7700,2.57,4160.0,54.03
1,On Time or Early,88653,4.29,8186.0,9.23


In [10]:
payment_analysis_query = """
SELECT
    p.payment_type,

    COUNT(*) AS payment_records,

    COUNT(
        DISTINCT p.order_id
    ) AS orders_using_method,

    ROUND(
        SUM(p.payment_value),
        2
    ) AS total_payment_value,

    ROUND(
        AVG(p.payment_value),
        2
    ) AS average_payment_value,

    ROUND(
        AVG(p.payment_installments),
        2
    ) AS average_installments

FROM payments AS p

INNER JOIN orders AS o
    ON p.order_id = o.order_id

WHERE o.order_status = 'delivered'

GROUP BY p.payment_type

ORDER BY total_payment_value DESC;
"""

payment_analysis_df = run_query(
    payment_analysis_query
)

payment_analysis_df

,payment_type,payment_records,orders_using_method,total_payment_value,average_payment_value,average_installments
0,credit_card,74586,74304,12101094.88,162.24,3.5
1,boleto,19191,19191,2769932.58,144.33,1.0
2,voucher,5493,3679,343013.19,62.45,1.0
3,debit_card,1486,1485,208421.12,140.26,1.0


In [1]:
# Final Business Insights

# 1. The dataset contains 96,477 paid and delivered orders from 93,357
#    unique customers. These orders generated approximately R$15.42 million,
#    with an average order value of R$159.86.

# 2. Monthly order volume and revenue grew strongly during 2017. November
#    2017 recorded the highest displayed monthly revenue at approximately
#    R$1.15 million.

# 3. Health and beauty generated the highest product revenue at approximately
#    R$1.23 million. Bed, bath and table recorded the highest item quantity
#    among the top categories.

# 4. São Paulo was the largest customer market, producing 40,500 delivered
#    orders and approximately R$5.77 million in payment revenue.

# 5. Alagoas recorded the highest late-delivery percentage at 23.93%.
#    However, delivery percentages should always be interpreted together
#    with order volume.

# 6. Delivery performance was strongly associated with customer satisfaction.
#    Late deliveries received an average review score of 2.57, compared with
#    4.29 for on-time or early deliveries.

# 7. Approximately 54.03% of late-delivery reviews were low scores, compared
#    with only 9.23% for on-time or early deliveries.

# 8. Credit cards were the dominant payment method, accounting for 74,304
#    delivered orders and approximately R$12.10 million in payment value.
#    Credit-card customers used an average of 3.5 installments.

# ## Analysis Limitations

# - The dataset covers a limited historical period and the earliest months
#   contain very small order volumes.
# - Product revenue uses item price and excludes freight charges.
# - Some states have small order counts, which can produce unstable percentages.
# - The relationship between delivery delay and review score is an association;
#   this analysis does not prove that delay is the only cause of low reviews.